## Part 1: introduction

Data import:

In [1]:
import pandas as pd

df = pd.read_csv("data/SPX_data_26.02.24.csv")

df.head()

,underlying_symbol,quote_datetime,root,expiration,strike,option_type,open,high,low,close,trade_volume,bid_size,bid,ask_size,ask,underlying_bid,underlying_ask,open_interest,time_to_maturity
0,^SPX,2026-02-24 16:15:00,SPX,2026-03-20,200.0,C,0.0,0.0,0.0,0.0,0,2,6673.0,1,6692.9,6852.66,6913.22,35,0.063014
1,^SPX,2026-02-24 16:15:00,SPX,2026-03-20,400.0,C,0.0,0.0,0.0,0.0,0,1,6473.3,1,6492.5,6852.66,6913.22,13,0.063014
2,^SPX,2026-02-24 16:15:00,SPX,2026-03-20,600.0,C,0.0,0.0,0.0,0.0,0,1,6273.5,1,6294.5,6852.66,6913.22,6,0.063014
3,^SPX,2026-02-24 16:15:00,SPX,2026-03-20,800.0,C,0.0,0.0,0.0,0.0,0,2,6074.7,1,6093.2,6852.66,6913.22,9,0.063014
4,^SPX,2026-02-24 16:15:00,SPX,2026-03-20,1000.0,C,0.0,0.0,0.0,0.0,0,10,5877.8,10,5892.1,6852.66,6913.22,1101,0.063014


In [ ]:
from src.black_scholes import implied_volatility

df['C_mkt'] = (df['bid'] + df['ask']) / 2
df['S_0'] = (df['underlying_bid'] + df['underlying_ask']) / 2
r_annual = 0  

# compute IV for the whole dataset
df['implied_vol'] = df.apply(
    lambda row: implied_volatility(row['C_mkt'], row['S_0'], row['strike'], row['time_to_maturity'], r_annual),
    axis=1
)

# clean the dataset
df_clean = df.dropna(subset=['implied_vol'])

In [ ]:
df_clean.to_csv('data/SPX_data_clean_IV.csv', index=False)

## Part 2: synthetic dataset creation

In [ ]:
import os
import numpy as np
import pandas as pd
import src.heston as hst
from src.black_scholes import implied_volatility
from tqdm import tqdm  

# reading dataset with IV and extract ranges for parameters
df_market = pd.read_csv('data/SPX_data_clean_IV.csv')
df_market['S0_mid'] = (df_market['underlying_bid'] + df_market['underlying_ask']) / 2
S0_mercato = df_market['S0_mid'].mean()

T_min, T_max = df_market['time_to_maturity'].min(), df_market['time_to_maturity'].max()
df_market['moneyness_real'] = df_market['strike'] / df_market['S0_mid']
moneyness_min, moneyness_max = df_market['moneyness_real'].min(), df_market['moneyness_real'].max()

df_atm = df_market[(df_market['moneyness_real'] >= 0.98) & (df_market['moneyness_real'] <= 1.02)]
variance_min = max(0.005, df_atm['implied_vol'].min() ** 2)
variance_max = min(0.25, df_atm['implied_vol'].max() ** 2)

# generate synthetic dataset
N_SAMPLES = 20000
dataset_rows = []

print(f"Synthetic data generation:")
for i in tqdm(range(N_SAMPLES), desc="Progress:"):
    kappa_casuale = np.random.uniform(0.1, 5.0)
    xi_casuale = np.random.uniform(0.05, 1.0)
    rho_casuale = np.random.uniform(-0.95, 0.0)  
    theta_casuale = np.random.uniform(variance_min, variance_max)
    V0_casuale = np.random.uniform(variance_min, variance_max)
    
    maturity_casuale = np.random.uniform(T_min, T_max)
    moneyness_casuale = np.random.uniform(moneyness_min, moneyness_max)
    strike_casuale = S0_mercato * moneyness_casuale
    
    try:
        # calculate Heston price
        price_heston = hst.heston_call_price(
            S0=S0_mercato, K=strike_casuale, T=maturity_casuale, r=r_annual,
            kappa=kappa_casuale, theta=theta_casuale, xi=xi_casuale, rho=rho_casuale, V0=V0_casuale
        )
        
        # convert Heston price to implied volatility
        target_iv = implied_volatility(price_heston, S0_mercato, strike_casuale, maturity_casuale, r_annual)
        
        if not np.isnan(target_iv) and target_iv > 0:
            dataset_rows.append([kappa_casuale, theta_casuale, xi_casuale, rho_casuale, V0_casuale, strike_casuale, maturity_casuale, target_iv])
    except:
        continue

# save new dataset
columns = ['kappa', 'theta', 'xi', 'rho', 'V0', 'strike', 'time_to_maturity', 'target_iv']
df_sintetico = pd.DataFrame(dataset_rows, columns=columns)
os.makedirs('data', exist_ok=True)
df_sintetico.to_csv('data/heston_synthetic_dataset.csv', index=False)

print(f"Samples generated: {len(df_sintetico)}.")

## Part 3: define and train NN

In [ ]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"  

import torch
import joblib
import src.nn_for_volatility as hst_nn

# Definiamo i percorsi dove sono salvati i pesi e lo scaler
PATH_MODELLO = 'model/heston_pytorch_model.pth'
PATH_SCALER = 'model/heston_pytorch_scaler.pkl'


if os.path.exists(PATH_MODELLO) and os.path.exists(PATH_SCALER):
    print("Uploading model:")
    
    model = hst_nn.heston_volatility_nn()
    model.load_state_dict(torch.load(PATH_MODELLO))
    model.eval()
    scaler = joblib.load(PATH_SCALER)
    
    print("Model uploaded.")

else:
    print("Model not found: training in progress.")
    
    # if the files don't exist, train the model and save it
    model, scaler = hst_nn.train_heston_model(
        csv_path='data/heston_synthetic_dataset.csv',
        epochs=60,
        batch_size=64,
        patience=5
    )
    print("Model trained and saved.")

## Part 4: calibration

In [ ]:
from scipy.optimize import minimize
import numpy as np

# Rileggiamo i dati di mercato reali salvati nella Part 1
df_market = pd.read_csv('data/SPX_data_clean_IV.csv')

# Recuperiamo i vincoli estratti dinamicamente dai dati ATM nella Part 2
# (Utilizziamo le stesse variabili variance_min e variance_max calcolate prima)
bounds = [
    (0.1, 5.0),                  # kappa
    (variance_min, variance_max), # theta (vincolato ai regimi di varianza reali)
    (0.05, 1.2),                 # xi
    (-0.95, 0.0),                # rho (rigorosamente negativo per l'azionario)
    (variance_min, variance_max)  # V0 (vincolato ai regimi di varianza reali)
]

# Vettore di partenza (Initial Guess) finanziariamente plausibile
initial_guess = [2.0, (variance_min + variance_max)/2, 0.3, -0.6, (variance_min + variance_max)/2]

# Esecuzione del problema di ottimizzazione (Minimi Quadrati - Slide 8)
result = minimize(
    fun=hst_nn.heston_pytorch_objective,
    x0=initial_guess,
    args=(df_market, model, scaler),
    method='L-BFGS-B',
    bounds=bounds,
    options={'disp': True, 'maxiter': 150}
)

# Estrazione dei risultati ottimali fittati dalla rete
if result.success:
    kappa_opt, theta_opt, xi_opt, rho_opt, V0_opt = result.x
    print("\n✅ CALIBRAZIONE COMPLETATA CON SUCCESSO!")
    print("-" * 50)
    print(f"Kappa (Mean Reversion Speed) : {kappa_opt:.4f}")
    print(f"Theta (Long-Term Variance)   : {theta_opt:.4f} (Vol equivalente: {np.sqrt(theta_opt)*100:.2f}%)")
    print(f"Xi    (Vol of Vol)           : {xi_opt:.4f}")
    print(f"Rho   (Leverage / Skew)      : {rho_opt:.4f}")
    print(f"V0    (Initial Variance)     : {V0_opt:.4f} (Vol equivalente: {np.sqrt(V0_opt)*100:.2f}%)")
    print("-" * 50)
    
    # Test della Condizione di Feller
    feller_soddisfatta = 2 * kappa_opt * theta_opt > xi_opt**2
    print(f"Condizione di Feller (2*kappa*theta > xi^2): {'RISPETTATA' if feller_soddisfatta else 'VIOLATA'}")
else:
    print(f"❌ Errore durante la convergenza: {result.message}")